In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(
    "/content/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "text"]
)
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
print(df.shape)
print(df["label"].value_counts())

(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64


In [4]:
df.to_csv("/content/sms_spam.csv", index=False)

In [ ]:
# from google.colab import files
# files.download("/content/sms_spam.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
df = pd.read_csv("/content/sms_spam.csv")

In [47]:
X = df['text']
y = df['label']

In [48]:
y = y.map({'ham': 0, 'spam': 1})

In [49]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [50]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [51]:
import re
def preprocess(text):
  text = text.lower()
  text = ''.join(c for c in text if c.isalnum() or c.isspace())
  words = text.split()
  words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
  return ' '.join(words)

X = X.apply(preprocess)

In [52]:
X.iloc[0]

'go jurong point crazy available bugis n great world la e buffet cine got amore wat'

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42 , stratify=y)

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)

In [56]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}

In [57]:
from sklearn.metrics import accuracy_score, f1_score

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    }

In [58]:
for model, result in results.items():
    print(model, result)

Naive Bayes {'accuracy': 0.9524663677130045, 'f1': 0.7836734693877551}
Logistic Regression {'accuracy': 0.9659192825112107, 'f1': 0.8538461538461538}
SVM {'accuracy': 0.9838565022421525, 'f1': 0.9370629370629371}
Random Forest {'accuracy': 0.967713004484305, 'f1': 0.8625954198473282}


In [59]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 0.5, 1, 2, 5, 10, 20]
}

grid = GridSearchCV(
    LinearSVC(),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

Best Parameters: {'C': 10}
Best CV F1: 0.922964834597783


In [60]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))

Test Accuracy: 0.9829596412556054
Test F1: 0.9337979094076655


In [61]:
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

grid_rf = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200, 300]},
             scoring='f1')

In [62]:
print("Best Parameters:", grid_rf.best_params_)
print("Best CV F1:", grid_rf.best_score_)

Best Parameters: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 300}
Best CV F1: 0.8873551410421026


In [63]:
best_rf = grid_rf.best_estimator_

y_pred_rf = best_rf.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Test F1:", f1_score(y_test, y_pred_rf))

Test Accuracy: 0.9704035874439462
Test F1: 0.8754716981132076
